In [ ]:
pip install pandas scikit-learn joblib matplotlib

In [ ]:
import os
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


# =========================
# 1. ĐỌC DỮ LIỆU
# =========================
DATA_PATH = "student_skill_trend_dataset_2000.csv"
MODEL_DIR = "saved_models"

os.makedirs(MODEL_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)

print("Kích thước dataset:", df.shape)
print("Các cột:")
print(df.columns.tolist())
print("\n5 dòng đầu:")
print(df.head())


# =========================
# 2. XÁC ĐỊNH LABEL / TARGET
# =========================
TARGETS = ["strong_skill", "weak_skill", "trend_label"]

for target in TARGETS:
    if target not in df.columns:
        raise ValueError(f"Không tìm thấy cột target: {target}")


# =========================
# 3. LOẠI BỎ CÁC CỘT KHÔNG DÙNG LÀM INPUT
# =========================
# Những cột metadata không nên đưa vào model
metadata_cols = ["sample_id", "user_id", "snapshot_date"]

# Nếu bạn muốn tránh leakage mạnh hơn cho weak_skill/strong_skill
# thì có thể bỏ thêm các cột accuracy theo skill ở đây.
# Tạm thời để nguyên cho bản demo.
# leakage_cols = [
#     "listening_accuracy_30d",
#     "speaking_accuracy_30d",
#     "reading_accuracy_30d",
#     "writing_accuracy_30d",
#     "vocabulary_accuracy_30d",
#     "grammar_accuracy_30d",
# ]
# metadata_cols += leakage_cols

drop_input_cols = [col for col in metadata_cols if col in df.columns]


# =========================
# 4. HÀM TRAIN 1 MODEL
# =========================
def train_one_model(dataframe: pd.DataFrame, target_col: str):
    # Không dùng các target khác làm input
    other_targets = [c for c in TARGETS if c != target_col]

    X = dataframe.drop(columns=other_targets + [target_col] + drop_input_cols, errors="ignore")
    y = dataframe[target_col].copy()

    print(f"\n========== TRAIN TARGET: {target_col} ==========")
    print("Số lượng feature:", X.shape[1])
    print("Tên feature:")
    print(X.columns.tolist())

    # Chia train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # Tách cột số và cột category
    numeric_features = X_train.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
    categorical_features = X_train.select_dtypes(include=["object", "bool"]).columns.tolist()

    print("\nNumeric features:", numeric_features)
    print("Categorical features:", categorical_features)

    # Pipeline cho cột số
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])

    # Pipeline cho cột category
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    # Gộp preprocessing
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    # Model
    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        class_weight="balanced"
    )

    # Full pipeline
    clf = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    # Train
    clf.fit(X_train, y_train)

    # Predict
    y_pred = clf.predict(X_test)

    # Đánh giá
    acc = accuracy_score(y_test, y_pred)
    print(f"\nAccuracy ({target_col}): {acc:.4f}")
    print("\nClassification report:")
    print(classification_report(y_test, y_pred))

    # Lưu model
    model_path = os.path.join(MODEL_DIR, f"{target_col}_model.pkl")
    joblib.dump(clf, model_path)
    print(f"Đã lưu model tại: {model_path}")

    return clf, X.columns.tolist()


# =========================
# 5. TRAIN 3 MODEL
# =========================
feature_map = {}

for target in TARGETS:
    model, feature_names = train_one_model(df, target)
    feature_map[target] = feature_names

# Lưu danh sách feature để dùng cho inference
joblib.dump(feature_map, os.path.join(MODEL_DIR, "feature_map.pkl"))
print("\nĐã lưu feature_map.pkl")

Kích thước dataset: (2000, 40)
Các cột:
['user_id', 'grade_id', 'role', 'coin', 'score', 'streak', 'days_since_last_study', 'vip_days_remaining', 'is_vip_active', 'unit_progress_percent', 'section_progress_percent', 'active_days_7d', 'active_days_14d', 'active_days_30d', 'lessons_completed_7d', 'lessons_completed_14d', 'lessons_completed_30d', 'questions_answered_7d', 'questions_answered_14d', 'questions_answered_30d', 'accuracy_7d', 'accuracy_14d', 'accuracy_30d', 'avg_attempt_count_7d', 'coins_earned_7d', 'coins_earned_30d', 'writing_eval_count_30d', 'speaking_eval_count_30d', 'avg_writing_ai_score_30d', 'avg_speaking_ai_score_30d', 'skip_item_quantity', 'listening_accuracy_30d', 'speaking_accuracy_30d', 'reading_accuracy_30d', 'writing_accuracy_30d', 'vocabulary_accuracy_30d', 'grammar_accuracy_30d', 'strong_skill', 'weak_skill', 'trend_label']

5 dòng đầu:
   user_id  grade_id  role  coin  score  streak  days_since_last_study  \
0        1        12  USER   217   1074      13      

In [ ]:
import joblib
import pandas as pd

MODEL_DIR = "saved_models"

# Load model
strong_model = joblib.load(f"{MODEL_DIR}/strong_skill_model.pkl")
weak_model = joblib.load(f"{MODEL_DIR}/weak_skill_model.pkl")
trend_model = joblib.load(f"{MODEL_DIR}/trend_label_model.pkl")

feature_map = joblib.load(f"{MODEL_DIR}/feature_map.pkl")

# Lấy danh sách feature theo model weak_skill
# 3 model hiện tại dùng chung input gần như giống nhau
expected_features = feature_map["weak_skill"]

# Ví dụ 1 học sinh mới
student_data = {
    "role": "USER",
    "coin": 120,
    "score": 850,
    "streak": 6,
    "days_since_last_study": 1,
    "vip_days_remaining": 20,
    "is_vip_active": 1,

    "grade_id": 7,
    "unit_id": 3,
    "section_id": 2,
    "lesson_id": 5,

    "unit_progress_percent": 55.0,
    "section_progress_percent": 60.0,

    "active_days_7d": 5,
    "active_days_14d": 9,
    "active_days_30d": 18,

    "lessons_completed_7d": 4,
    "lessons_completed_14d": 7,
    "lessons_completed_30d": 12,

    "questions_answered_7d": 20,
    "questions_answered_14d": 38,
    "questions_answered_30d": 80,

    "accuracy_7d": 71.5,
    "accuracy_14d": 69.0,
    "accuracy_30d": 67.5,

    "avg_attempt_count_7d": 1.4,
    "coins_earned_7d": 18,
    "coins_earned_30d": 55,

    "writing_eval_count_30d": 2,
    "speaking_eval_count_30d": 1,
    "avg_writing_ai_score_30d": 63.0,
    "avg_speaking_ai_score_30d": 58.0,

    "has_skip_item": 1,

    "listening_accuracy_30d": 55.0,
    "speaking_accuracy_30d": 52.0,
    "reading_accuracy_30d": 71.0,
    "writing_accuracy_30d": 62.0,
    "vocabulary_accuracy_30d": 75.0,
    "grammar_accuracy_30d": 68.0,
}

# Tạo DataFrame
X_new = pd.DataFrame([student_data])

# Nếu thiếu cột nào thì tự thêm 0
for col in expected_features:
    if col not in X_new.columns:
        X_new[col] = 0

# Giữ đúng thứ tự cột
X_new = X_new[expected_features]

# Predict
pred_strong = strong_model.predict(X_new)[0]
pred_weak = weak_model.predict(X_new)[0]
pred_trend = trend_model.predict(X_new)[0]

print("Kết quả dự đoán:")
print("Strong skill :", pred_strong)
print("Weak skill   :", pred_weak)
print("Trend label  :", pred_trend)

Kết quả dự đoán:
Strong skill : vocabulary
Weak skill   : speaking
Trend label  : STABLE
